# Robot Harness Optimization

<p align="center">
  <img src="images/rho_thumb.png" width="900">
</p>

[RHO](https://arxiv.org/abs/2606.16458) (*Your Coding Agent is Secretly a Roboticist*) optimizes robot policy **source code** before deployment, so the deployed system runs frozen Python with no language model in the control loop. It instantiates this with **HELIX**, which treats a whole multi-file repository as the evolving candidate and uses a tool-enabled coding agent as the mutation operator.

## Goals

* Understand the evolution configuration: the objective, the editable surface, the protected evaluator, and the acceptance gate
* Read a recorded run across two robot tasks and six validation trials, and work out how much of it was useful
* Watch the seed and the deployed policy run the same trial live
* Learn how to set up your own RHO experiment
* Learn how to deploy an RHO-evolved policy on the Robosuite benchmark tasks

## What this notebook reads

An evolution needs a large coding model and hours of search, so this notebook is
a walkthrough of an experiment recorded ahead of time rather than a run of one.
The code that produced it is here to read, and the deployed policy is here to
run live at the end.

The cell below reports which pieces of that recording are on disk: the study
report, the seed repository, the repository the search deployed, and the
configuration files that defined the run.

In [ ]:
import json
import sys
from pathlib import Path
from time import perf_counter

import pandas as pd
from IPython.display import display

sys.path.insert(0, "/ryzers/notebooks/scripts")

# capx_demo chdir()s into the CaP-X tree on import, so every path in this
# notebook stays absolute. None of these imports touch the GPU.
import capx_demo
import rho_demo
import rho_multitask_demo
import rho_report

RECORDED_ROOT = Path("/ryzers/notebooks/recorded_results")
SEED_REPO = RECORDED_ROOT / "repos" / "seed"
BEST_REPO = RECORDED_ROOT / "repos" / "selected"

# Keep live rollout videos out of the read-only recorded tree.
rho_demo.VIDEO_ROOT = Path("/tmp/rho_live_videos")

STATUS = rho_report.preflight(RECORDED_ROOT)
print(rho_report.format_preflight(STATUS))

## What RHO actually edits

The evolving candidate is a whole repository, divided into an editable policy surface and a protected evaluation boundary.

Two task policies sit on the editable surface. Each one is a real program that a local Gemma E4B model produced during a CaP-X run, kept with its original failure intact so that the mutator has something genuine to diagnose:

| Policy | Task | How the seed fails |
| --- | --- | --- |
| `solver/tasks/cube_stack.py` | stack a red cube on a green cube | indexes a flat XYZ pose as if it were nested, raising `IndexError` after the grasp |
| `solver/tasks/cube_lift.py` | pick up the red cube and lift it clear of the table | calls `numpy.array` without importing numpy, raising `NameError` |

Each failure is drawn from a real sweep rather than written by hand.

`solver/geometry.py` and `solver/runtime.py` start out as empty modules. The mutator may add shared helpers there, and either policy may import them. Stacking and lifting both need grasp and pose geometry, so there is real shared structure to factor out, which is what makes this a multi-file search rather than two independent single-file repairs.

In [ ]:
rho_report.require_ready(STATUS)

import tomllib

SEED_CONFIG = tomllib.loads((SEED_REPO / "helix.toml").read_text())
PROTECTED = set(SEED_CONFIG["evaluator"]["protected_files"]) | {"helix.toml"}

surface = []
for path in sorted(SEED_REPO.rglob("*")):
    if not path.is_file() or "__pycache__" in path.parts:
        continue
    relative = path.relative_to(SEED_REPO).as_posix()
    if relative in PROTECTED:
        role = "protected — mutations that touch it are rejected"
    elif relative.startswith("solver/"):
        role = "EDITABLE — the search space"
    else:
        role = "reference"
    surface.append(
        {
            "file": relative,
            "role": role,
            "lines": len(path.read_text().splitlines()),
        }
    )

display(pd.DataFrame(surface).set_index("file"))

## The experiment definition

`helix.toml` is the entire experiment, printed below without filtering. Most of the design work in an evolution run goes into two string fields rather than into the numeric settings: `objective` states what a good repository looks like, and `agent.background` tells the mutator how to behave inside the loop. Those two fields move the outcome more than any number in the file.

Among the numeric settings, two change the character of the search more than the rest. `acceptance_criterion = "strict_improvement"` decides which children survive their parent, since a child that regresses on its training minibatch is discarded before it ever reaches validation. `frontier_type = "instance"` decides what the population keeps, retaining the best candidate for each validation trial rather than collapsing onto a single best-on-average one. Nearly everything else is budget: how many generations to run, how many proposals to draw per generation, and a hard cap on evaluations that applies regardless.

In [ ]:
print((SEED_REPO / "helix.toml").read_text())

## Running a full evolution

Loading a large model and running evolutions would take more time than this session has, so the commands below are printed rather than run. To reproduce the recorded experiment yourself, the first fetches the model into Lemonade; the second runs the search.

To watch a single generation instead of the whole search, pass `--generations 1` and `--proposals 1`. One proposal is small enough to follow end to end: the coding agent reads the evaluator's diagnostics, edits the repository, and the gate scores the result against its parent. Expect several minutes and expect it to sometimes accept nothing, which is a normal outcome. The recorded run below rejected four of its eight proposals.

In [ ]:
print("===== 1. fetch the 30B mutation model (18 GB, not in this image) =====")
print(rho_report.PULL_MODEL_COMMAND)
print()
print("===== 2. the command that produced the recorded study =====")
print(rho_report.REGENERATE_COMMAND)

## Protection against reward hacking

An agent that can edit its own scorer will eventually optimize the scorer instead of the policy. The mutator and the evaluator are separated by three mechanisms.

1. **Protected files are hashed.** At run start HELIX writes SHA-256 digests of
   `probe.py`, `helix.toml`, `opencode.json`, `CONTRACT.md`, `scenarios.json`
   and `provenance.json` into `.helix/evaluator_manifest.json`. A candidate that
   modifies any of them is rejected *before* it is scored, so editing the
   benchmark is not a viable strategy.
2. **Tool permissions are allow-listed.** `opencode.json` denies edits outside
   `solver/`, denies web fetch and search, denies sub-agent and skill tools, and
   permits exactly two shell commands: a compile check and the self-check probe.
3. **Scores come back over a fixed protocol.** `probe.py` prints one
   `HELIX_RESULT=[[score, side_info], ...]` line, positionally matched to the
   example ids HELIX wrote into `helix_batch.json`. The `side_info` payload
   carries reward, completion, traceback and evaluator feedback, and that is
   what becomes the reflective prompt for the next mutation.

The mutator never sees the raw `HELIX_RESULT` line, only the diagnostics.

In [ ]:
print("===== the evaluator the candidate repository runs =====")
print((SEED_REPO / "probe.py").read_text())

print("===== what the mutator is allowed to do =====")
permissions = json.loads((SEED_REPO / "opencode.json").read_text())["permission"]
print(json.dumps({key: permissions[key] for key in ("edit", "bash")}, indent=2))

---

## A recorded full run

Everything from here on reads a study recorded ahead of time: the same two tasks, the same seed repository, and the same mutator, run for two generations with four proposals each. The cell below loads it and reports what it cost.

In [ ]:
REPORT = rho_report.load_report(RECORDED_ROOT)
COST = rho_report.study_cost(REPORT)
TASKS = rho_report.validation_tasks(REPORT)
DEPLOYED = REPORT["selected_candidate"]

print(f"Tasks:            {', '.join(TASKS)}")
print(f"Mutation model:   {COST['mutation_model']}")
print(f"Generations:      {COST['generations']}"
      f" · {COST['proposal_slots_per_generation']} proposal slots each")
print(f"Deployed:         {DEPLOYED}")
print(f"Evolution time:   {COST['evolution_seconds'] / 60:.1f} min")
print(f"Total study time: {COST['total_seconds'] / 60:.1f} min")
print()
print(rho_report.selection_rule(REPORT))

## Search progress

Selection deploys the candidate with the highest mean validation reward, so plotting that mean against generation shows what the search was actually buying. Every proposal that cleared the training gate appears as a dot, the line tracks the best score reached so far, and the crosses along the bottom count the proposals the gate rejected in each generation.

Expect the line to jump on the first generation and then flatten. With a 30B mutator reading a real traceback, an `IndexError` from a mis-indexed pose and a `NameError` from a missing import are close to mechanical repairs, and the search finds them immediately.

The table underneath gives the run as lineage: which parent each candidate came from, which files it touched, what the gate decided, and the mean validation reward it reached. The seed sits at the top on 0.000, since it crashes on all six trials, and the proposals the gate rejected have no score at all because they were discarded before validation ever ran.

Read a 1.000 carefully. Each of those scores is a single evaluation, and replaying the same six trials five times each puts the deployed policy at 19 of 30. The score says the repair works, not how often it succeeds, and those six trials are the ones the search optimized against, so it says nothing about how far the repair generalizes.

In [ ]:
rho_report.plot_progress(REPORT)

PROGRESS = rho_report.progress_frame(REPORT)
print(f"Proposals made:     {len(PROGRESS) - 1}")
print(f"Cleared the gate:   {int(PROGRESS['validated'].sum()) - 1}")
print(f"Rejected:           {int((~PROGRESS['validated']).sum())}")
print()
display(rho_report.lineage_frame(REPORT))

## What the deployed repository changed

The diff below compares the deployed candidate against the seed, split by file. This is the artifact RHO ships. A reviewer can read it line by line and accept or reject it, which a weight delta does not allow. Nothing in it calls a language model, so the deployed system runs the same way every time.

In [ ]:
display(rho_report.diff_summary(REPORT["selected_diff"]))

for name, section in rho_report.diff_sections(REPORT["selected_diff"]).items():
    print(f"\n===== {name} =====")
    print(section)

## Start the robot services

The replay below runs in simulation, which needs the perception and control
stack loaded first: OWLv2 for text-conditioned box grounding, SAM2 for masking,
Contact-GraspNet for grasp proposals, and PyRoKi for IK.

Note that a language model is never called here, the deployed policy is frozen
Python.


In [ ]:
SETUP_STARTED = perf_counter()
rho_demo.ensure_services(model=None)
print(f"Services ready in {perf_counter() - SETUP_STARTED:.1f}s")


## Watch both policies run

The cell below runs the two frozen repositories live, one validation trial per task, so you can watch the seed fail and the deployed policy finish.

Grasp sampling is not deterministic, so the deployed policy sometimes misses.

In [ ]:
print("Both runs execute frozen Python: no generation, no tool loop, no LLM.")

for task in TASKS:
    trial = rho_report.validation_trial(REPORT, task)
    live_rollouts = []
    print()
    print(f"===== {task.replace('_', ' ')} · validation trial {trial} =====")
    for label, repo in (("seed", SEED_REPO), (f"deployed ({DEPLOYED})", BEST_REPO)):
        print(f"\n-- {label} --")
        started = perf_counter()
        result = rho_multitask_demo.replay_trials(
            repo,
            trials={task: [trial]},
            capture=True,
            progress=rho_report.rollout_narrator(),
        )[0]
        result["wall_seconds"] = perf_counter() - started
        print(f"   ({result['wall_seconds']:.1f}s)")
        live_rollouts.append((label, result))

    print()
    display(
        pd.DataFrame(
            [
                {
                    "policy": label,
                    "reward": result["reward"],
                    "raw reward": result.get("raw_reward"),
                    "solved": result["task_completed"],
                    "seconds": round(result["wall_seconds"], 1),
                }
                for label, result in live_rollouts
            ]
        ).set_index("policy")
    )
    capx_demo.show_comparison_grid(live_rollouts)

## Taking the deployed repository with you

The deployed candidate is a directory of Python on this machine, so it outlives the notebook.

In [ ]:
print(f"Deployed repository: {BEST_REPO}")
for path in sorted(BEST_REPO.rglob("*.py")):
    print(f"  {path.relative_to(BEST_REPO)}")

print()
print("===== replay it on any trial =====")
print(f"""import sys; sys.path.insert(0, "/ryzers/notebooks/scripts")
import rho_multitask_demo

rho_multitask_demo.replay_trials(
    "{BEST_REPO}",
    trials={{"cube_stack": [11, 12, 13]}},   # any trial ids you like
)""")

---

## Key takeaways

* **The candidate is a whole repository.** RHO evolves several source files with imports between them, so a mutation can add a shared helper in one file and call it from another. What ships is code a reviewer can read.
* **Deployment runs no model.** The replays above execute frozen Python against the robot services. The language model cost is paid up front, during the search, rather than on every rollout as in notebook 3.
* **The training gate does most of the filtering.** A child must strictly improve on its parent's minibatch before it is validated at all. Most proposals fail that test, and those rejections are where the bulk of the compute goes.

## What to try next

* Replay the deployed repository on trial ids the search never used, with the snippet printed above, and see how much of the validation result survives.
* Add a third task by dropping a policy into `solver/tasks/` and adding it to the `TASKS` registry, then check whether the mutator starts factoring shared code into `solver/geometry.py`. Scan the new seed across several trials first: a seed that already succeeds gives the search nothing to work with.
* Pull the mutation model with the first command above and run the study with `--generations 1 --proposals 1`, then read the agent's diff alongside the evaluator feedback it was given and judge whether the edit follows from the diagnostics.

## References

* [RHO: Your Coding Agent is Secretly a Roboticist](https://arxiv.org/abs/2606.16458) — the paper this notebook follows
* [Code as Policies](https://code-as-policies.github.io/) — the policy-as-source-code framing the seed programs build on
* `scripts/rho_multitask_demo.py` — the experiment definition used here
* `scripts/rho_report.py` — the reporting and plotting helpers

---

## Teardown

The cell below stops the model and robotics services this notebook started. Run it last, since the comparisons above need them.

In [ ]:
rho_demo.stop_owned_services()
print("Notebook-owned model and robotics services stopped.")